In [24]:
import pandas as pd

train_df = pd.read_csv("../Data/train_2000_2020.csv")

train_df = train_df.drop(columns=["Date", "Time"])

TARGET = "Thunderstorm_24h"

X_train = train_df.drop(columns=[TARGET])

In [25]:
import joblib

cat_model = joblib.load("../Data/CatBoost_Final_Research_Model.pkl")

In [26]:
df = pd.read_csv("../Data/test_2023_2025.csv")

# Remove unused columns
df = df.drop(columns=["Date", "Time"])

TARGET = "Thunderstorm_24h"

X_test = df.drop(columns=[TARGET])

In [27]:
import pandas as pd
import numpy as np

# ==========================================================
# CATBOOST PROBABILITY ANALYSIS
# ==========================================================

# Predict probabilities
test_prob = cat_model.predict_proba(X_test)[:, 1]

# Copy test data
analysis_df = X_test.copy()

analysis_df["Probability"] = test_prob

# ==========================================================
# Create 7 equally populated probability groups
# ==========================================================

analysis_df["Probability Level"] = pd.qcut(
    analysis_df["Probability"],
    q=7,
    labels=[
        "<35% (Very Low)",
        "45% (Low)",
        "55% (Moderately Low)",
        "65% (Moderate)",
        "75% (Moderately High)",
        "85% (High)",
        "95% (Very High)"
    ]
)

# ==========================================================
# Choose representative features
# ==========================================================

top_features = [

    "PW",

    "KI",

    "CAPE",

    "LI",

    "CT",

    "SWEAT",

    "SI"

]

# ==========================================================
# Median values
# ==========================================================

summary = (

    analysis_df

    .groupby("Probability Level")[top_features]

    .median()

)

# ==========================================================
# Number of samples in each level
# ==========================================================

sample_count = (

    analysis_df

    .groupby("Probability Level")

    .size()

    .rename("Samples")

)

summary = summary.join(sample_count)

# ==========================================================
# Order from highest to lowest
# ==========================================================

order = [

    "95% (Very High)",

    "85% (High)",

    "75% (Moderately High)",

    "65% (Moderate)",

    "55% (Moderately Low)",

    "45% (Low)",

    "<35% (Very Low)"

]

summary = summary.reindex(order)

summary = summary.round(2)

# ==========================================================
# Display
# ==========================================================

print("\nCatBoost Thunderstorm Probability Levels\n")

display(summary)


CatBoost Thunderstorm Probability Levels



,PW,KI,CAPE,LI,CT,SWEAT,SI,Samples
Probability Level,,,,,,,,
95% (Very High),64.61,37.90,2605.41,-5.10,20.8,239.0,-1.27,199
85% (High),60.27,36.35,2539.69,-4.92,19.7,234.5,-0.57,199
75% (Moderately High),59.52,35.90,2474.66,-4.39,19.3,246.7,-0.27,199
65% (Moderate),46.26,31.35,2330.47,-3.96,17.5,197.9,1.45,198
55% (Moderately Low),35.60,24.60,1762.67,-4.10,14.1,142.4,4.01,199
45% (Low),26.84,2.00,262.54,-0.68,12.1,118.3,7.19,199
<35% (Very Low),21.31,-16.90,0.00,4.77,7.9,69.0,12.23,199


In [28]:
import pandas as pd
import numpy as np

# ==========================================================
# CATBOOST OPERATIONAL THUNDERSTORM THRESHOLDS
# ==========================================================

# Predict probabilities
test_prob = cat_model.predict_proba(X_test)[:, 1]

analysis_df = X_test.copy()
analysis_df["Probability"] = test_prob

# ==========================================================
# Probability Levels (Equal Frequency Groups)
# ==========================================================

analysis_df["Probability Level"] = pd.qcut(
    analysis_df["Probability"],
    q=7,
    labels=[
        "<35% (Very Low)",
        "45% (Low)",
        "55% (Moderately Low)",
        "65% (Moderate)",
        "75% (Moderately High)",
        "85% (High)",
        "95% (Very High)"
    ]
)

# ==========================================================
# Representative Features
# ==========================================================

higher_features = [
    "PW",
    "KI",
    "CAPE_V",
    "CT",
    "SWEAT"
]

lower_features = [
    "LI",
    "SI"
]

order = [
    "95% (Very High)",
    "85% (High)",
    "75% (Moderately High)",
    "65% (Moderate)",
    "55% (Moderately Low)",
    "45% (Low)",
    "<35% (Very Low)"
]

rows = []

for level in order:

    subset = analysis_df[
        analysis_df["Probability Level"] == level
    ]

    row = {"Probability Level": level}

    # Higher value → Higher thunderstorm likelihood
    for feature in higher_features:

        value = subset[feature].quantile(0.25)

        if feature == "CAPE" and value <= 1:
            row[feature] = "≈0"
        else:
            row[feature] = f"≥ {value:.2f}"

    # Lower value → Higher thunderstorm likelihood
    for feature in lower_features:

        value = subset[feature].quantile(0.75)

        row[feature] = f"≤ {value:.2f}"

    rows.append(row)

threshold_table = pd.DataFrame(rows)

# ==========================================================
# Display
# ==========================================================

print("\nOperational Thunderstorm Thresholds Derived from CatBoost\n")

display(threshold_table)

# ==========================================================
# Save
# ==========================================================

threshold_table.to_csv(
    "CatBoost_Operational_Thunderstorm_Thresholds.csv",
    index=False
)

print("Table Saved Successfully!")


Operational Thunderstorm Thresholds Derived from CatBoost



,Probability Level,PW,KI,CAPE_V,CT,SWEAT,LI,SI
0,95% (Very High),≥ 60.46,≥ 35.60,≥ 2325.28,≥ 19.80,≥ 223.40,≤ -3.51,≤ -0.25
1,85% (High),≥ 55.43,≥ 34.10,≥ 2196.30,≥ 18.90,≥ 216.20,≤ -3.47,≤ 0.40
2,75% (Moderately High),≥ 50.59,≥ 32.90,≥ 1799.40,≥ 17.90,≥ 215.85,≤ -2.85,≤ 0.79
3,65% (Moderate),≥ 41.59,≥ 26.75,≥ 1540.90,≥ 15.10,≥ 157.45,≤ -2.08,≤ 3.08
4,55% (Moderately Low),≥ 29.40,≥ 16.35,≥ 976.69,≥ 10.35,≥ 83.40,≤ -1.54,≤ 6.55
5,45% (Low),≥ 22.54,≥ -10.60,≥ 56.39,≥ 7.05,≥ 59.40,≤ 1.45,≤ 10.14
6,<35% (Very Low),≥ 17.35,≥ -26.70,≥ 0.00,≥ -7.00,≥ 39.84,≤ 6.36,≤ 16.01


Table Saved Successfully!


In [29]:
import os
import pickle
import numpy as np
import pandas as pd

In [30]:
os.makedirs("models", exist_ok=True)
os.makedirs("thresholds", exist_ok=True)

In [32]:
with open("models/CatBoost_Final.pkl", "wb") as f:
    pickle.dump(cat_model, f)

print("Model saved.")


Model saved.


In [33]:
feature_order = list(X_train.columns)

with open("models/feature_order.pkl", "wb") as f:
    pickle.dump(feature_order, f)

print(feature_order)

['SI', 'LI', 'LI_V', 'SWEAT', 'KI', 'CT', 'VT', 'TT', 'CAPE', 'CAPE_V', 'CIN', 'CIN_V', 'EL', 'EL_V', 'LFC', 'LFC_V', 'BRN', 'BRN_V', 'LCL_T', 'LCL_P', 'THETAE_LCL', 'MML_PT', 'MML_MR', 'THK_1000_500', 'PW']


In [35]:
feature_importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": cat_model.get_feature_importance()

})

feature_importance = feature_importance.sort_values(

    "Importance",

    ascending=False

).reset_index(drop=True)

feature_importance

,Feature,Importance
0,PW,24.793223
1,MML_PT,8.774427
2,THETAE_LCL,7.464047
3,KI,6.439376
4,TT,5.020140
5,LCL_T,4.965786
6,SWEAT,4.681135
7,LCL_P,4.192091
8,LFC,4.188094
9,THK_1000_500,3.673334


In [36]:
feature_dict = dict(

    zip(

        feature_importance["Feature"],

        feature_importance["Importance"]

    )

)

with open("models/feature_importance.pkl","wb") as f:

    pickle.dump(feature_dict,f)

print("Feature importance saved.")

Feature importance saved.


In [39]:
best_threshold = 0.32

In [41]:
import joblib
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Load model
model = joblib.load("../Data/CatBoost_Final_Research_Model.pkl")

# Load data
df = pd.read_csv("../Data/test_2023_2025.csv")

# Remove unused columns
df = df.drop(columns=["Date", "Time"])

TARGET = "Thunderstorm_24h"

X_test = df.drop(columns=[TARGET])
y_test = df[TARGET]

In [42]:
# ============================================================
# Probability Analysis on Test Set
# ============================================================

import pandas as pd
import numpy as np

# X_test -> Test features
# y_test -> True labels
# model -> Trained CatBoost model

# Predict probabilities
y_prob = cat_model.predict_proba(X_test)[:, 1] * 100

# Predict classes using your chosen threshold
y_pred = (y_prob >= best_threshold).astype(int)

# Create comparison dataframe
results = X_test.copy()

results["Actual"] = y_test.values
results["Predicted Probability (%)"] = np.round(y_prob, 2)
results["Predicted Class"] = y_pred

print("=" * 80)
print("Probability Statistics")
print("=" * 80)

print(f"Minimum Probability : {y_prob.min():.2f}%")
print(f"Maximum Probability : {y_prob.max():.2f}%")
print(f"Mean Probability    : {y_prob.mean():.2f}%")
print(f"Median Probability  : {np.median(y_prob):.2f}%")

print("\n")

print("=" * 80)
print("Top 20 Highest Probabilities")
print("=" * 80)

display(
    results.sort_values(
        "Predicted Probability (%)",
        ascending=False
    ).head(20)
)

print("\n")

print("=" * 80)
print("Top 20 Lowest Probabilities")
print("=" * 80)

display(
    results.sort_values(
        "Predicted Probability (%)",
        ascending=True
    ).head(20)
)

print("\n")

print("=" * 80)
print("Actual Thunderstorm Cases")
print("=" * 80)

display(
    results[
        results["Actual"] == 1
    ].sort_values(
        "Predicted Probability (%)",
        ascending=False
    )
)

print("\n")

print("=" * 80)
print("Actual Non-Thunderstorm Cases")
print("=" * 80)

display(
    results[
        results["Actual"] == 0
    ].sort_values(
        "Predicted Probability (%)",
        ascending=False
    )
)

print("\n")

print("=" * 80)
print("Probability Distribution")
print("=" * 80)

bins = [0,10,20,30,40,50,60,70,80,90,100]

distribution = pd.cut(
    y_prob,
    bins=bins,
    include_lowest=True
).value_counts().sort_index()

print(distribution)


Probability Statistics
Minimum Probability : 0.36%
Maximum Probability : 77.56%
Mean Probability    : 18.65%
Median Probability  : 13.75%


Top 20 Highest Probabilities


,SI,LI,LI_V,SWEAT,KI,CT,VT,TT,CAPE,CAPE_V,...,LCL_T,LCL_P,THETAE_LCL,MML_PT,MML_MR,THK_1000_500,PW,Actual,Predicted Probability (%),Predicted Class
1134,-2.620920,-4.757193,-5.127455,257.8,41.5,22.4,22.7,45.1,1968.029541,2740.489384,...,298.575698,987.638116,361.745147,301.750763,19.352494,NaN,73.958913,0,77.56,1
386,-2.343637,-5.287691,-5.744437,239.0,40.0,22.1,23.5,45.6,2535.570360,3273.041075,...,298.404132,983.609139,361.572950,302.111950,17.807509,5768.515760,69.765843,1,73.23,1
534,-2.518793,-6.112817,-7.008048,267.4,35.9,22.1,23.7,45.8,3159.724085,3842.587132,...,299.226449,988.875233,364.940429,302.294725,19.312533,5793.171689,62.890245,0,69.19,1
1244,-2.118199,-4.848354,-5.407365,250.2,38.0,22.0,24.1,46.1,1953.493541,2591.260796,...,297.654735,984.498348,357.642794,301.329875,18.045816,5765.175992,64.262169,1,68.32,1
404,-3.412213,-3.257637,-3.692014,246.2,41.9,21.1,28.1,49.2,948.862792,1448.677953,...,297.677621,988.772596,357.025251,306.179479,15.800680,NaN,65.528218,1,68.03,1
1028,-1.286894,-2.842225,-3.096778,268.4,39.8,21.0,21.7,42.7,1179.833754,1911.461153,...,297.952705,982.387008,359.491922,301.204328,18.560536,NaN,73.943053,0,67.27,1
1164,-1.908713,-5.757455,-6.156000,235.0,38.4,21.5,23.3,44.8,2821.062663,3616.565565,...,298.852677,976.676698,365.163351,302.763384,19.086065,NaN,70.050317,1,67.19,1
1190,-1.795235,-7.266715,-7.807576,247.4,38.8,21.4,23.5,44.9,4951.734055,5967.555532,...,299.801323,979.742780,369.762251,302.558309,18.783160,5779.822989,61.054089,1,66.44,1
441,-2.060661,-5.460415,-5.910901,281.0,40.8,21.9,23.1,45.0,2418.464325,3193.691648,...,298.676624,984.832755,362.766201,302.631318,17.829197,NaN,70.294069,0,65.43,1
1184,-0.140155,-2.498750,-3.084211,242.4,37.9,20.3,20.5,40.8,1733.511982,2511.203223,...,298.077225,988.780457,359.008324,301.729707,18.505503,NaN,67.240078,0,64.44,1




Top 20 Lowest Probabilities


,SI,LI,LI_V,SWEAT,KI,CT,VT,TT,CAPE,CAPE_V,...,LCL_T,LCL_P,THETAE_LCL,MML_PT,MML_MR,THK_1000_500,PW,Actual,Predicted Probability (%),Predicted Class
685,17.885659,6.652955,5.436970,53.0,-33.7,-11.9,23.1,11.2,0.000000,0.0,...,291.065661,991.139945,329.103554,296.208603,9.863832,5764.751746,14.528940,0,0.36,1
1374,12.007157,6.367282,5.164158,89.8,-25.7,9.9,17.9,27.8,0.000000,0.0,...,291.000406,978.964855,330.542209,296.018869,11.192738,5756.686482,22.306405,0,0.36,1
731,8.522985,4.774758,3.545414,133.4,-19.5,12.3,20.3,32.6,0.000000,0.0,...,292.463384,987.262694,334.753264,298.290601,11.967483,5773.427816,24.659628,0,0.36,1
34,11.513758,5.429344,4.160257,114.4,-16.1,9.7,17.7,27.4,0.000000,0.0,...,292.891007,983.890427,336.878535,298.898211,7.778728,5753.677835,18.097755,0,0.36,1
1376,17.184150,6.711507,5.495580,28.0,-35.1,-1.3,19.7,18.4,0.000000,0.0,...,290.800917,978.953044,329.835359,296.455580,9.967455,5757.968872,16.159365,0,0.36,1
244,12.894418,6.704305,5.545899,64.0,5.3,6.3,20.3,26.6,0.000000,0.0,...,290.830146,976.556782,330.269733,296.021519,11.967053,5754.099015,25.421318,0,0.37,1
723,17.251414,5.940620,4.791623,27.0,-30.1,-11.9,24.1,12.2,0.000000,0.0,...,292.216182,986.294990,333.959857,297.676015,11.685970,5765.031815,18.279320,0,0.37,1
1364,18.773119,7.214093,6.026870,65.0,-41.5,-12.9,22.1,9.2,0.000000,0.0,...,290.525163,982.335191,328.406135,296.294851,11.715651,5748.911597,16.563032,0,0.37,1
1303,18.685659,6.069969,4.873905,55.0,-52.5,-22.7,22.3,-0.4,0.000000,0.0,...,290.841855,964.144072,332.049106,297.429284,10.303290,5762.724163,14.687193,0,0.37,1
717,15.910739,6.352983,5.472370,49.0,-52.9,-23.7,25.3,1.6,0.000000,0.0,...,289.723006,960.709033,328.592856,297.443966,10.427296,5760.360150,15.512542,0,0.38,1




Actual Thunderstorm Cases


,SI,LI,LI_V,SWEAT,KI,CT,VT,TT,CAPE,CAPE_V,...,LCL_T,LCL_P,THETAE_LCL,MML_PT,MML_MR,THK_1000_500,PW,Actual,Predicted Probability (%),Predicted Class
386,-2.343637,-5.287691,-5.744437,239.0,40.0,22.1,23.5,45.6,2535.570360,3273.041075,...,298.404132,983.609139,361.572950,302.111950,17.807509,5768.515760,69.765843,1,73.23,1
1244,-2.118199,-4.848354,-5.407365,250.2,38.0,22.0,24.1,46.1,1953.493541,2591.260796,...,297.654735,984.498348,357.642794,301.329875,18.045816,5765.175992,64.262169,1,68.32,1
404,-3.412213,-3.257637,-3.692014,246.2,41.9,21.1,28.1,49.2,948.862792,1448.677953,...,297.677621,988.772596,357.025251,306.179479,15.800680,NaN,65.528218,1,68.03,1
1164,-1.908713,-5.757455,-6.156000,235.0,38.4,21.5,23.3,44.8,2821.062663,3616.565565,...,298.852677,976.676698,365.163351,302.763384,19.086065,NaN,70.050317,1,67.19,1
1190,-1.795235,-7.266715,-7.807576,247.4,38.8,21.4,23.5,44.9,4951.734055,5967.555532,...,299.801323,979.742780,369.762251,302.558309,18.783160,5779.822989,61.054089,1,66.44,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,5.421901,-9.298069,-10.359339,51.8,16.5,10.1,30.1,40.2,3969.454899,4939.565655,...,298.158700,968.393762,363.023442,303.152060,14.668483,5758.297953,36.014816,1,6.71,1
223,1.240390,4.581517,4.366254,258.2,36.0,19.6,21.7,41.3,0.000000,0.000000,...,288.076418,879.491687,334.660865,299.501074,13.376738,5740.731490,60.067874,1,6.22,1
1237,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,...,188.928176,75.307679,395.577185,NaN,NaN,NaN,0.002729,1,5.13,1
1335,7.269972,2.873458,1.797692,165.0,-5.5,14.7,18.5,33.2,40.833211,25.961450,...,292.957031,963.832952,340.114154,298.014580,13.680146,5749.599769,34.606217,1,1.13,1




Actual Non-Thunderstorm Cases


,SI,LI,LI_V,SWEAT,KI,CT,VT,TT,CAPE,CAPE_V,...,LCL_T,LCL_P,THETAE_LCL,MML_PT,MML_MR,THK_1000_500,PW,Actual,Predicted Probability (%),Predicted Class
1134,-2.620920,-4.757193,-5.127455,257.8,41.5,22.4,22.7,45.1,1968.029541,2740.489384,...,298.575698,987.638116,361.745147,301.750763,19.352494,NaN,73.958913,0,77.56,1
534,-2.518793,-6.112817,-7.008048,267.4,35.9,22.1,23.7,45.8,3159.724085,3842.587132,...,299.226449,988.875233,364.940429,302.294725,19.312533,5793.171689,62.890245,0,69.19,1
1028,-1.286894,-2.842225,-3.096778,268.4,39.8,21.0,21.7,42.7,1179.833754,1911.461153,...,297.952705,982.387008,359.491922,301.204328,18.560536,NaN,73.943053,0,67.27,1
441,-2.060661,-5.460415,-5.910901,281.0,40.8,21.9,23.1,45.0,2418.464325,3193.691648,...,298.676624,984.832755,362.766201,302.631318,17.829197,NaN,70.294069,0,65.43,1
1184,-0.140155,-2.498750,-3.084211,242.4,37.9,20.3,20.5,40.8,1733.511982,2511.203223,...,298.077225,988.780457,359.008324,301.729707,18.505503,NaN,67.240078,0,64.44,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,8.522985,4.774758,3.545414,133.4,-19.5,12.3,20.3,32.6,0.000000,0.000000,...,292.463384,987.262694,334.753264,298.290601,11.967483,5773.427816,24.659628,0,0.36,1
685,17.885659,6.652955,5.436970,53.0,-33.7,-11.9,23.1,11.2,0.000000,0.000000,...,291.065661,991.139945,329.103554,296.208603,9.863832,5764.751746,14.528940,0,0.36,1
34,11.513758,5.429344,4.160257,114.4,-16.1,9.7,17.7,27.4,0.000000,0.000000,...,292.891007,983.890427,336.878535,298.898211,7.778728,5753.677835,18.097755,0,0.36,1
1374,12.007157,6.367282,5.164158,89.8,-25.7,9.9,17.9,27.8,0.000000,0.000000,...,291.000406,978.964855,330.542209,296.018869,11.192738,5756.686482,22.306405,0,0.36,1




Probability Distribution
(-0.001, 10.0]    609
(10.0, 20.0]      195
(20.0, 30.0]      193
(30.0, 40.0]      197
(40.0, 50.0]      119
(50.0, 60.0]       58
(60.0, 70.0]       19
(70.0, 80.0]        2
(80.0, 90.0]        0
(90.0, 100.0]       0
Name: count, dtype: int64


In [43]:
print("Best Threshold =", best_threshold)

raw_prob = model.predict_proba(X_test)[:,1]

print("Min raw probability :", raw_prob.min())
print("Max raw probability :", raw_prob.max())

Best Threshold = 0.32
Min raw probability : 0.003566563225251034
Max raw probability : 0.7756187817953937


In [44]:
raw_prob = model.predict_proba(X_test)[:, 1]

y_pred = (raw_prob >= best_threshold).astype(int)

results = pd.DataFrame({
    "Actual": y_test.values,
    "Raw Probability": raw_prob,
    "Probability (%)": raw_prob * 100,
    "Predicted Class": y_pred
})

print("Threshold =", best_threshold)

display(
    results.sort_values(
        "Probability (%)",
        ascending=True
    ).head(20)
)

Threshold = 0.32


,Actual,Raw Probability,Probability (%),Predicted Class
685,0,0.003567,0.356656,0
1374,0,0.003604,0.360431,0
1376,0,0.003608,0.360810,0
34,0,0.003620,0.361957,0
731,0,0.003624,0.362403,0
1364,0,0.003658,0.365834,0
723,0,0.003681,0.368112,0
1303,0,0.003700,0.370039,0
244,0,0.003716,0.371644,0
729,0,0.003773,0.377304,0


In [45]:
display(
    results.loc[
        results["Probability (%)"] < 1,
        ["Probability (%)", "Predicted Class"]
    ].head(20)
)

,Probability (%),Predicted Class
6,0.659365,0
8,0.565498,0
9,0.496112,0
10,0.489751,0
11,0.427358,0
12,0.517992,0
13,0.494868,0
14,0.495162,0
16,0.623880,0
17,0.762717,0


In [46]:
highest = results.loc[
    results["Probability (%)"].idxmax()
]

print(highest)

Actual              0.000000
Raw Probability     0.775619
Probability (%)    77.561878
Predicted Class     1.000000
Name: 1134, dtype: float64


In [47]:
display(
    results.sort_values(
        "Probability (%)",
        ascending=False
    ).head(1)
)

,Actual,Raw Probability,Probability (%),Predicted Class
1134,0,0.775619,77.561878,1


In [48]:
display(
    X_test.loc[[highest.name]]
)

,SI,LI,LI_V,SWEAT,KI,CT,VT,TT,CAPE,CAPE_V,...,LFC_V,BRN,BRN_V,LCL_T,LCL_P,THETAE_LCL,MML_PT,MML_MR,THK_1000_500,PW
1134,-2.62092,-4.757193,-5.127455,257.8,41.5,22.4,22.7,45.1,1968.029541,2740.489384,...,786.177341,37.145507,51.725274,298.575698,987.638116,361.745147,301.750763,19.352494,NaN,73.958913


In [49]:
train_row = X_test.loc[[highest.name]].T
train_row.columns = ["Training"]

live_row = feature_df.T
live_row.columns = ["Live"]

compare = train_row.join(live_row)
compare["Difference"] = compare["Live"] - compare["Training"]

display(compare.round(4))

NameError: name 'feature_df' is not defined

In [52]:
import pandas as pd

# qcut boundaries
_, bins = pd.qcut(
    analysis_df["Probability"],
    q=7,
    retbins=True,
    duplicates="drop"
)

mapping = pd.DataFrame({
    "Display_Percentage": [95, 85, 75, 65, 55, 45, 35],
    "Min_Model_Probability": [
        bins[6],
        bins[5],
        bins[4],
        bins[3],
        bins[2],
        bins[1],
        bins[0]
    ]
})

display(mapping)

mapping.to_csv(
    "CatBoost_Operational_Probability_Mapping.csv",
    index=False
)

print("Operational probability mapping saved successfully!")

,Display_Percentage,Min_Model_Probability
0,95,0.399631
1,85,0.296162
2,75,0.194069
3,65,0.093043
4,55,0.028071
5,45,0.008310
6,35,0.003567


Operational probability mapping saved successfully!
